In [1]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.4 MB/s eta 0:00:00


In [2]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import string

In [3]:
# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
# Import Sastrawi untuk stemming bahasa Indonesia
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    sastrawi_available = True
except ImportError:
    print("Sastrawi tidak tersedia. Stemming akan dilewati.")
    sastrawi_available = False

In [5]:
file_path = "/content/TomLembong123.csv"
df = pd.read_csv(file_path)

In [6]:
print("Data awal:")
print(df.head())

Data awal:
            publishedAt    authorDisplayName  \
0  2025-08-01T14:49:47Z        @ferryirwandi   
1  2025-08-01T14:53:42Z  @MFDOfficialChannel   
2  2025-08-01T14:55:39Z             @Frenysf   
3  2025-08-01T14:59:40Z     @fathurrajli2495   
4  2025-08-01T14:59:59Z      @AntiPembodohan   

                                         textDisplay  likeCount  
0  Hallo warga sipil sekalian! jadi cemana menuru...        524  
1                             Strategi politik, basi         43  
2              Labil, plinpan, basi, dan ah sudahlah         28  
3                                     Ya, begitulah!          6  
4  Hanya TOM seorang yg layak mengapresiasi aboli...         27  


In [7]:
#PREPROCESSING
def preprocessing(text):
    if pd.isna(text):
        return ""

    #Hapus tag HTML
    text = re.sub(r'<.*?>', ' ', text)

    #Hapus URL (http, https, www)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    #Hapus mention (@username) dan hashtag (#topik)
    text = re.sub(r'@\w+|#\w+', '', text)

    #Hapus emoji (berbagai kategori simbol Unicode)
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F700-\U0001F77F"  # alchemical symbols
                               u"\U0001F780-\U0001F7FF"  # geometric shapes
                               u"\U0001F800-\U0001F8FF"  # supplemental arrows
                               u"\U0001F900-\U0001F9FF"  # pictographs
                               u"\U0001FA00-\U0001FA6F"  # chess symbols
                               u"\U0001FA70-\U0001FAFF"  # extended pictographs
                               u"\U00002702-\U000027B0"  # dingbats
                               u"\U000024C2-\U0001F251"  # enclosed characters
                               "]+", flags=re.UNICODE)
    text = emoji_pattern.sub('', text)

    #Case Folding → ubah ke huruf kecil
    text = text.lower()

    #Hapus karakter khusus dan angka
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  #hanya huruf & spasi
    text = re.sub(r'\d+', '', text)            #hapus angka

    #Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['clean_text'] = df['textDisplay'].apply(preprocessing)

print("\n=== Contoh hasil setelah preprocessing ===")
print(df[['textDisplay', 'clean_text']].head(10))



=== Contoh hasil setelah preprocessing ===
                                         textDisplay  \
0  Hallo warga sipil sekalian! jadi cemana menuru...   
1                             Strategi politik, basi   
2              Labil, plinpan, basi, dan ah sudahlah   
3                                     Ya, begitulah!   
4  Hanya TOM seorang yg layak mengapresiasi aboli...   
5                        Waaah presiden ku pahlawan🥰   
6  Up juga terkait penangkapan pemain judol yang ...   
7  Gmn bg?<br>aturan2 aneh yg banyak dikaji dan d...   
8  hati-hati kejaksaan, blunder lagi kena rujak h...   
9  Pintar nya pak Prabowo dan tim,.<br>Jadikan Jo...   

                                          clean_text  
0  hallo warga sipil sekalian jadi cemana menurut...  
1                              strategi politik basi  
2                 labil plinpan basi dan ah sudahlah  
3                                       ya begitulah  
4  hanya tom seorang yg layak mengapresiasi aboli...  
5        

In [8]:
# Normalization Dictionary
normalization_dict = {

    "gk": "tidak", "nggak": "tidak", "tdk": "tidak", "ga": "tidak", "gak": "tidak",
    "bukan": "tidak", "nggk": "tidak", "ngga": "tidak",
    "bgt": "banget", "bngt": "banget", "bener": "benar", "bnr": "benar",
    "keren": "bagus", "mantap": "bagus", "top": "bagus",
    "dr": "dari", "yg": "yang", "dgn": "dengan", "dg": "dengan",
    "krn": "karena", "krna": "karena", "jd": "jadi", "jdi": "jadi",
    "lg": "lagi", "ud": "sudah", "udh": "sudah", "dah": "sudah",
    "org": "orang", "orng": "orang", "tau": "tahu", "tau": "tahu",
    "kyk": "seperti", "kyak": "seperti", "gmn": "bagaimana", "gimana": "bagaimana",
    "knp": "kenapa", "kenapa": "mengapa", "emg": "memang", "emang": "memang",
    "bisa": "dapat", "bs": "bisa", "tp": "tapi", "tp": "tetapi",
    "mantul": "bagus sekali", "kece": "bagus", "oke": "baik",
    "recommended": "direkomendasikan", "rekomended": "direkomendasikan",
    "zonk": "buruk", "payah": "buruk", "ancur": "buruk sekali",
    "parah": "buruk sekali", "jelek": "buruk"
}

def normalize(text):
    words = text.split()
    normalized_words = []
    for word in words:
        if word in normalization_dict:
            normalized_words.append(normalization_dict[word])
        else:
            found = False
            for abbrev, full in normalization_dict.items():
                if len(word) > 2 and abbrev in word:
                    normalized_words.append(full)
                    found = True
                    break
            if not found:
                normalized_words.append(word)
    return ' '.join(normalized_words)

df['clean_text'] = df['clean_text'].apply(normalize)
print("\nSetelah Normalization:")
print(df['clean_text'].head())


Setelah Normalization:
0    hallo tidak sipil sekalian jadi cemana menurut...
1                                strategi politik basi
2                      labil plinpan basi dan ah sudah
3                                         ya begitulah
4    hanya tom seorang yang layak tidak abolisi tid...
Name: clean_text, dtype: object


In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


# Stopword Removal
stop_factory = StopWordRemoverFactory()
default_stopwords = set(stop_factory.get_stop_words())

# custom stopwords
custom_stopwords = {
    'ada', 'adalah', 'adanya', 'akan', 'aku', 'anda', 'antara', 'apa', 'atau', 'bagi',
    'bahwa', 'baik', 'banyak', 'bisa', 'boleh', 'bukan', 'dalam', 'dan', 'dari', 'dengan',
    'di', 'dia', 'gak', 'hal', 'hanya', 'hingga', 'ia', 'ini', 'itu', 'jadi', 'jangan', 'jika',
    'juga', 'kalau', 'kami', 'kamu', 'karena', 'ke', 'kita', 'lagi', 'lain', 'lalu', 'mereka',
    'nya', 'oleh', 'pada', 'para', 'saat', 'saja', 'sama', 'saya', 'sebagai', 'secara', 'seperti',
    'setelah', 'sudah', 'tapi', 'tanpa', 'telah', 'tentang', 'tetapi', 'tidak', 'untuk', 'yang', 'yg',
    'yg', 'utk', 'cuman', 'deh', 'Btw', 'tapi', 'gua', 'gue', 'lo', 'lu', 'kalo', 'trs', 'jd', 'nih', 'ntar', 'nya',
    'lg', 'gk', 'ecusli', 'dpt', 'dr', 'kpn', 'kok', 'kyk', 'donk', 'yah', 'u', 'ya', 'ga', 'km', 'eh', 'sih', 'eh',
    'bang', 'br', 'kyk', 'rp', 'jt', 'kan', 'gpp', 'sm', 'usah', 'mas', 'sob', 'thx', 'ato', 'jg', 'gw', 'wkwk', 'mak',
    'haha', 'iy', 'k', 'tp', 'haha', 'dg', 'dri', 'duh', 'ye', 'wkwkwk', 'syg', 'btw', 'nerjemahin', 'gaes', 'guys', 'moga',
    'kmrn', 'nemu', 'yukkk', 'wkwkw', 'klas', 'iw', 'ew', 'lho', 'sbnry', 'org', 'gtu', 'bwt', 'klrga', 'clau', 'lbh',
    'cpet', 'ku', 'wke', 'mba', 'mas', 'sdh', 'kmrn', 'oi', 'spt', 'dlm', 'bs', 'krn', 'jgn', 'sapa', 'spt', 'sh', 'wakakaka',
    'sihhh', 'hehe', 'ih', 'dgn', 'la', 'kl', 'ttg', 'mana', 'kmna', 'kmn', 'tdk', 'tuh', 'dah', 'kek', 'ko', 'pls', 'bbrp', 'pd', 'mah', 'dhhh',
    'kpd', 'tuh', 'kzl', 'byar', 'si', 'sii', 'cm', 'sy', 'hahahaha', 'weh', 'dlu', 'tuhh'
}

all_stopwords = default_stopwords.union(custom_stopwords)

def remove_stopwords(text):
    words = text.split()
    sentiment_words = {'tidak', 'bukan', 'jangan', 'buruk', 'bagus', 'baik', 'jelek'}
    return ' '.join([word for word in words if word not in all_stopwords or word in sentiment_words])

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
print("\nSetelah Stopword Removal:")
print(df['clean_text'].head())


Setelah Stopword Removal:
0             hallo tidak sipil sekalian cemana kalian
1                                strategi politik basi
2                                labil plinpan basi ah
3                                            begitulah
4    tom seorang layak tidak abolisi tidak siapapun...
Name: clean_text, dtype: object


In [ ]:
# Stemming
stemmer = StemmerFactory().create_stemmer()
df['clean_text'] = df['clean_text'].apply(lambda x: stemmer.stem(x))
print("\nSetelah Stemming:")
print(df['clean_text'].head())

# Remove empty texts
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)


Setelah Stemming:
0               hallo tidak sipil sekali cemana kalian
1                                strategi politik basi
2                                labil plinpan basi ah
3                                               begitu
4    tom orang layak tidak abolisi tidak siapa tom ...
Name: clean_text, dtype: object


In [ ]:
df.head()

,publishedAt,authorDisplayName,textDisplay,likeCount,clean_text
0,2025-08-01T14:49:47Z,@ferryirwandi,Hallo warga sipil sekalian! jadi cemana menuru...,524,hallo tidak sipil sekali cemana kalian
1,2025-08-01T14:53:42Z,@MFDOfficialChannel,"Strategi politik, basi",43,strategi politik basi
2,2025-08-01T14:55:39Z,@Frenysf,"Labil, plinpan, basi, dan ah sudahlah",28,labil plinpan basi ah
3,2025-08-01T14:59:40Z,@fathurrajli2495,"Ya, begitulah!",6,begitu
4,2025-08-01T14:59:59Z,@AntiPembodohan,Hanya TOM seorang yg layak mengapresiasi aboli...,27,tom orang layak tidak abolisi tidak siapa tom ...


In [ ]:
df.to_csv('TomLembong-praprepo1.csv', index=False)